# Kaggle: baseline eval -> SFT (QLoRA) -> evaluate

Runs, in order, against the real `Qwen/Qwen2.5-3B-Instruct` model (per `config.yaml`'s `model.active`):

1. **Zero-shot baseline eval** on the held-out test split (`src/eval/generate.py` with no adapter).
2. **SFT training** (`src/train.train_sft`, QLoRA 4-bit, hyperparameters read directly from `config.yaml` -- not re-typed here, so this notebook can't drift from the config the rest of the repo uses).
3. **`sft_qlora` eval** on the same test split, with the trained adapter attached.
4. Score both runs with the deterministic rubric (`src/eval/score.py`) and append to `results/summary.csv`.

Same scripts as the local smoke test (`train_sft.py`) plus the `src/eval/generate.py` / `score.py` pair validated locally against the smoke adapter beforehand -- see `LOG.md` 2026-08-17. Only the model size, data volume, and GPU differ.

**Before running:**
1. Zip the repo root's `src/` folder *and* `config.yaml` together (this notebook reads hyperparameters straight out of `config.yaml`, so it needs to be inside the zip, not just `src/`).
2. Upload that zip, plus `data/splits/sft_train.jsonl`, `data/splits/sft_valid.jsonl`, `data/splits/sft_test.jsonl`, as a Kaggle Dataset.
3. Attach the dataset (Add Data), turn on a GPU accelerator (T4 x1 is enough -- a 3B model in 4-bit does not need multi-GPU, and `device_map={"":0}` pins to one GPU deliberately, see `src/eval/generate.py` docstring for why), Internet on (to pull the base model from HF).
4. Run all cells. A small `--limit`/`--max-steps` dry run happens automatically before the full run, matching this project's established pattern of proving the path/config work cheaply before committing GPU hours to the full run.

**Output:** `/kaggle/working/adapters/sft_qlora/` (download and place at `adapters/sft_qlora/` locally), `/kaggle/working/results/` (baseline + sft_qlora scored JSONL and `summary.csv` -- merge into local `results/`).

Resumable: `generate.py` skips ids already present in its output file, so a re-run after a Kaggle session timeout continues rather than restarting. Training itself is not resumable mid-run (3 epochs over 5,987 examples is expected to comfortably fit in one Kaggle session at this model size -- see runtime note in the training cell).

In [ ]:
!pip install -q -U "trl==1.10.0" "peft==0.20.0" "transformers==5.15.0" "bitsandbytes==0.50.1" "accelerate==1.14.0" pyyaml
# peft 0.20.0 requires torchao>=0.16.0 for an internal LoRA-dispatch check;
# Kaggle's base image ships torchao==0.10.0. Only bites when loading a PEFT
# adapter onto a full-precision (non-4-bit) base model -- not the case in
# this notebook's own eval calls, but uninstalled defensively anyway since
# this project doesn't use torchao at all and it cost a full 3h15m training
# run + eval crash to find in kaggle_sft_lora_fp.ipynb. See LOG.md 2026-08-19.
!pip uninstall -y -q torchao
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Unsloth (accelerated QLoRA path)

Added after the vanilla trl/bitsandbytes path measured ~117-125s/step on a single-GPU-pinned T4 (expected 2-8s/step) with no root cause found -- see `LOG.md` 2026-08-18. Unsloth uses custom Triton kernels specifically built for this "3B-7B QLoRA on a single Kaggle/Colab T4" scenario.

**Couldn't be fully validated locally** before this notebook was first written -- a bare `pip install unsloth` on the local dev machine silently upgraded `torch` and broke CUDA, then a second, more careful `--no-deps` attempt still hit a deeper transitive conflict (`torchao` needing a newer `torch` than the local CUDA setup uses). `--no-deps` also turned out to matter on Kaggle itself, not just locally: a bare (non-`--no-deps`) install broke CUDA on a real Kaggle session (the `dpo_from_sft` notebook's first run, 2026-08-19) even though an earlier Kaggle run with the same bare command hadn't -- non-deterministic across sessions, not a one-off local quirk. `--no-deps unsloth unsloth_zoo` is the fix now used everywhere. The CUDA-survived assertion right after the install stays regardless, as a safety net -- catches a broken install in seconds instead of after a long dry/full run.

In [ ]:
!pip install -q --no-deps unsloth unsloth_zoo
import torch
print("CUDA available after unsloth install:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), (
    "CUDA broke after installing unsloth -- exactly what happened locally (see LOG.md 2026-08-18) "
    "and non-deterministically on a real Kaggle session even with --no-deps once already (see "
    "LOG.md 2026-08-19). Do not proceed with --use-unsloth training if this assertion fails; fall "
    "back to the vanilla path (train_sft_args' use_unsloth=False) instead."
)

In [ ]:
import os, sys, zipfile

def find_repo(root="/kaggle/input"):
    """Locate the src/ package and config.yaml independently, by content
    marker, rather than assuming they sit at a fixed relative position to
    each other. Kaggle's /kaggle/input nesting -- and how different zip
    tools handle "zip these items together" vs. "zip this one folder" --
    has proven unpredictable across uploads on this account (see
    kaggle_teacher_gen.ipynb LOG.md 2026-08-16 note; this notebook hit a
    second instance of the same class of problem when config.yaml landed
    *inside* src/ instead of beside it, see LOG.md 2026-08-17). Marker:
    build_irac.py inside a directory named data/ whose parent is named
    src/ -- specific enough not to false-match anything else, and doesn't
    care how many wrapper folders surround it."""
    repo_dir = None
    config_path = None
    zip_path = None
    for r, dirs, files in os.walk(root):
        if "build_irac.py" in files and os.path.basename(r) == "data":
            src_dir = os.path.dirname(r)
            if os.path.basename(src_dir) == "src":
                repo_dir = os.path.dirname(src_dir)
        if config_path is None and "config.yaml" in files:
            config_path = os.path.join(r, "config.yaml")
        if "src.zip" in files:
            zip_path = os.path.join(r, "src.zip")
    return repo_dir, config_path, zip_path

repo_dir, config_path, zip_path = find_repo()

if repo_dir is None and zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/repo")
    repo_dir, config_path, _ = find_repo("/kaggle/working/repo")
    print("Extracted", zip_path)

if repo_dir is None or config_path is None:
    raise FileNotFoundError(
        f"repo_dir={repo_dir}, config_path={config_path} -- could not find both "
        "src/data/build_irac.py and config.yaml under /kaggle/input (in any "
        "nesting). Check the dataset is attached. If just attached/updated, "
        "try Restart & Run All."
    )

REPO_DIR = repo_dir
sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print("config.yaml at", config_path)

import yaml
with open(config_path) as f:
    cfg = yaml.safe_load(f)

MODEL_ID = cfg["model"]["candidates"][cfg["model"]["active"]]["hf_id"]
print("Active model:", MODEL_ID)

In [ ]:
import os

def find_data_file(name, root="/kaggle/input"):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(f"{name} not found under {root} -- check it was included in the uploaded dataset.")

TRAIN_FILE = find_data_file("sft_train.jsonl")
VALID_FILE = find_data_file("sft_valid.jsonl")
TEST_FILE = find_data_file("sft_test.jsonl")
print(TRAIN_FILE, VALID_FILE, TEST_FILE, sep="\n")

RESULTS_DIR = "/kaggle/working/results"
ADAPTERS_DIR = "/kaggle/working/adapters"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)
SUMMARY_CSV = os.path.join(RESULTS_DIR, "summary.csv")

# Imported here, not in the baseline-eval cells below, specifically so the
# baseline section (## 1.) can be skipped/commented out on a resumed run
# (e.g. after baseline already succeeded and only training needs a retry)
# without the sft_qlora eval/score cells further down silently losing
# these names. See LOG.md 2026-08-18.
from src.eval.generate import run as generate_run
from src.eval.score import score_file, append_summary_row

# The dry-run cells below only prove batch_size=8 fits in memory. The full
# runs use GEN_BATCH_SIZE=16 for throughput -- untested at this model size
# until the first full-run cell actually executes. If it OOMs, drop this to
# 8-12 and re-run (generate.py is resumable, so nothing already-written is
# lost) rather than assuming 16 is safe just because 8 was.
GEN_BATCH_SIZE = 16

## 1. Baseline (zero-shot) eval

Dry run on 8 rows first (catches a broken path/model-id/OOM cheaply), then the full 803-row test split.

In [ ]:
generate_run(
    input_path=TEST_FILE,
    output_path=os.path.join(RESULTS_DIR, "baseline_gen_dryrun.jsonl"),
    model_id=MODEL_ID,
    adapter_path=None,
    load_in_4bit=False,
    batch_size=8,
    max_new_tokens=350,
    limit=8,
)

In [ ]:
import json

with open(os.path.join(RESULTS_DIR, "baseline_gen_dryrun.jsonl"), encoding="utf-8") as f:
    dryrun_rows = [json.loads(l) for l in f]
print(dryrun_rows[0]["raw_response"][:400])
assert all(r["raw_response"].strip().startswith("Issue:") for r in dryrun_rows), (
    "dry-run output doesn't start cleanly with 'Issue:' -- check the prompt template "
    "boundary before spending GPU hours on the full run (see LOG.md 2026-08-17 "
    "completion-masking bug)."
)
print("Dry run looks sane -- proceeding to the full baseline eval.")

In [ ]:
BASELINE_GEN = os.path.join(RESULTS_DIR, "baseline_gen.jsonl")

generate_run(
    input_path=TEST_FILE,
    output_path=BASELINE_GEN,
    model_id=MODEL_ID,
    adapter_path=None,
    load_in_4bit=False,
    batch_size=GEN_BATCH_SIZE,
    max_new_tokens=350,
    limit=None,
)

In [ ]:
BASELINE_SCORED = os.path.join(RESULTS_DIR, "baseline_scored.jsonl")
baseline_summary = score_file(BASELINE_GEN, BASELINE_SCORED)
append_summary_row("baseline", baseline_summary, SUMMARY_CSV)
print("baseline:", baseline_summary)

## 2. SFT training (QLoRA, `sft_qlora` run)

Hyperparameters come from `config.yaml`'s `peft` and `training.sft` sections -- not re-typed here, so this can't silently drift from what the rest of the repo believes the run used. A short `--max-steps` dry run proves the training path works before committing to the full 3-epoch run.

In [ ]:
peft_cfg = cfg["peft"]
sft_cfg = cfg["training"]["sft"]
max_seq_length = cfg["training"]["max_seq_length"]

SFT_QLORA_DIR = os.path.join(ADAPTERS_DIR, "sft_qlora")

# Set False to fall back to the vanilla (validated-locally, but ~15-20x
# slower on Kaggle) path -- e.g. if the CUDA-survived-install assertion
# above failed. See LOG.md 2026-08-18.
USE_UNSLOTH = True

def train_sft_args(output_dir, max_steps=None, num_epochs=None):
    args = [
        "--model", MODEL_ID,
        "--train-file", TRAIN_FILE,
        "--eval-file", VALID_FILE,
        "--output-dir", output_dir,
        "--load-in-4bit",
        "--lora-r", str(peft_cfg["lora_r"]),
        "--lora-alpha", str(peft_cfg["lora_alpha"]),
        "--lora-dropout", str(peft_cfg["lora_dropout"]),
        "--target-modules", *peft_cfg["target_modules"],
        "--max-seq-length", str(max_seq_length),
        "--per-device-batch-size", str(sft_cfg["per_device_batch_size"]),
        "--gradient-accumulation-steps", str(sft_cfg["gradient_accumulation_steps"]),
        "--learning-rate", str(sft_cfg["learning_rate"]),
    ]
    if USE_UNSLOTH:
        args += ["--use-unsloth"]
    if max_steps is not None:
        args += ["--max-steps", str(max_steps)]
    else:
        args += ["--num-epochs", str(num_epochs)]
    return args

print(train_sft_args(SFT_QLORA_DIR, num_epochs=sft_cfg["epochs"]))

In [ ]:
import subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
# Python fully buffers stdout when it isn't a tty (true for a subprocess
# piped through subprocess.run), so training-step log lines can sit in the
# buffer and all get flushed to the Kaggle log at once -- makes per-step
# timing look instantaneous/simultaneous instead of showing real deltas.
# Unbuffered output gives honest timestamps for the ETA math. See LOG.md
# 2026-08-18.
env["PYTHONUNBUFFERED"] = "1"

dryrun_dir = os.path.join(ADAPTERS_DIR, "sft_qlora_dryrun")
cmd = [sys.executable, "-m", "src.train.train_sft"] + train_sft_args(dryrun_dir, max_steps=5)
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

In [ ]:
# Full training run. IMPORTANT: check the dry-run cell's per-step timing
# above before running this -- multiply it by ~1,125 (5,987 rows / effective
# batch 16 * 3 epochs) for a real ETA on this specific session, rather than
# trusting any estimate written here. Prior attempts on the vanilla path
# measured 117-125s/step even after fixing the multi-GPU pipeline-
# parallelism bug (single-GPU pin confirmed via the GPU-memory diagnostic
# lines), well over the 12-hour session cap -- unexplained, which is why
# --use-unsloth exists. If the dry run above still looks anywhere near that
# slow, stop here and report the timing rather than let this cell run for
# hours toward a guaranteed timeout. See LOG.md 2026-08-18.
cmd = [sys.executable, "-m", "src.train.train_sft"] + train_sft_args(SFT_QLORA_DIR, num_epochs=sft_cfg["epochs"])
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

## 3. `sft_qlora` eval

Same test split as the baseline, same generation script, adapter attached this time.

In [ ]:
SFT_QLORA_GEN = os.path.join(RESULTS_DIR, "sft_qlora_gen.jsonl")

generate_run(
    input_path=TEST_FILE,
    output_path=SFT_QLORA_GEN,
    model_id=MODEL_ID,
    adapter_path=SFT_QLORA_DIR,
    load_in_4bit=True,
    batch_size=GEN_BATCH_SIZE,
    max_new_tokens=350,
    limit=None,
)

In [ ]:
SFT_QLORA_SCORED = os.path.join(RESULTS_DIR, "sft_qlora_scored.jsonl")
sft_qlora_summary = score_file(SFT_QLORA_GEN, SFT_QLORA_SCORED)
append_summary_row("sft_qlora", sft_qlora_summary, SUMMARY_CSV)
print("sft_qlora:", sft_qlora_summary)

In [ ]:
import pandas as pd
df = pd.read_csv(SUMMARY_CSV)
print(df.to_string(index=False))